<a href="https://colab.research.google.com/github/victorealeal/pressao-hospitalar-sns/blob/main/02_limpeza_dados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Limpeza e padronização dos dados
Notebook 02: preparação das tabelas de ocupação e urgência para a análise.

## Metodologia

**Acumulação.** Os dias de internamento e os atendimentos da urgência são acumulados desde janeiro. Foi verificado em todas as instituições que o valor só diminui em casos raros (2 na ocupação e 22 na urgência), o que confirma o acumulado.

**Valores mensais.** O valor de cada mês é o acumulado menos o acumulado do mês anterior. Em janeiro é o próprio valor. Se o mês anterior estiver ausente, o valor do mês fica ausente.

**Taxa de ocupação mensal.** Dias de internamento do mês ÷ (camas × dias do mês) × 100, com camas = Lotação Praticada da linha. Conferida manualmente numa instituição (ULS de Matosinhos).

**Correspondência de 2024.** Em janeiro de 2024 muitas instituições mudaram de nome. Cada nome antigo foi ligado ao atual e validado comparando o número de camas antes e depois (43 instituições, nenhuma com variação superior a 30%). Limitação: o teste não distingue instituições de dimensão parecida e a correspondência não foi confirmada por fonte oficial.

**Critérios para ausências e valores anormais.**
1. Valor igual ao do mês anterior, incompatível com um acumulado: marcado como ausente.
2. Valor mensal negativo: marcado como ausente.
3. Nenhum valor é estimado ou preenchido, e um vazio não é tratado como zero.
4. Taxa mensal fora de 40% a 130%: mantida, mas marcada como pouco confiável. Os limites servem para sinalizar, e não para corrigir.
5. Os arquivos CSV originais nunca são alterados.

In [1]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import glob

base = '/content/drive/MyDrive/semantix/**/'
ocup = pd.read_csv(glob.glob(base + '*ocupa*.csv', recursive=True)[0], sep=None, engine='python', encoding='utf-8-sig')
urg = pd.read_csv(glob.glob(base + '*manchester*.csv', recursive=True)[0], sep=None, engine='python', encoding='utf-8-sig')

print(ocup.shape, urg.shape)

(2464, 7) (2200, 11)


## 1. Conversão de tipos
O período é convertido de texto para data, e são criadas as colunas de ano e mês.

In [3]:
ocup['Período'] = pd.to_datetime(ocup['Período'])
urg['Período'] = pd.to_datetime(urg['Período'])

ocup['ano'] = ocup['Período'].dt.year
ocup['mes'] = ocup['Período'].dt.month
urg['ano'] = urg['Período'].dt.year
urg['mes'] = urg['Período'].dt.month

print(ocup['Período'].dtype, ocup['Período'].min(), ocup['Período'].max())

datetime64[ns] 2022-01-01 00:00:00 2026-07-01 00:00:00


## 2. Padronização de nomes
Correção de dois nomes de instituição escritos de forma diferente nas duas tabelas.

In [4]:
ocup['Instituição'] = ocup['Instituição'].str.strip().replace({
    'Centro Hospitalar Universitário do Algarve,EPE': 'Centro Hospitalar Universitário do Algarve, EPE',
    'Centro Hospitalar Universitário de Lisboa Norte, EPE': 'Centro Hospitalar Universitário Lisboa Norte, EPE'})
urg['Instituição'] = urg['Instituição'].str.strip()

print(ocup['Instituição'].nunique(), urg['Instituição'].nunique())
print(len(set(ocup['Instituição']) - set(urg['Instituição'])))
print(len(set(urg['Instituição']) - set(ocup['Instituição'])))

85 74
11
0


## 3. Correspondência entre nomes antigos e novos
Em janeiro de 2024, muitas instituições mudaram de nome. Cada nome antigo é ligado ao nome atual, numa nova coluna `instituicao`.

In [5]:
correspondencia = {
    'Centro Hospitalar Barreiro/Montijo, EPE': 'Unidade Local de Saúde do Arco Ribeirinho, EPE',
    'Centro Hospitalar Entre Douro e Vouga, EPE': 'Unidade Local de Saúde de Entre Douro e Vouga, EPE',
    'Centro Hospitalar Médio Tejo, EPE': 'Unidade Local de Saúde do Médio Tejo, EPE',
    'Centro Hospitalar Póvoa de Varzim/Vila do Conde, EPE': 'Unidade Local de Saúde da Póvoa de Varzim/Vila do Conde, EPE',
    'Centro Hospitalar Tondela-Viseu, EPE': 'Unidade Local de Saúde de Dão-Lafões, EPE',
    'Centro Hospitalar Trás-os-Montes e Alto Douro, EPE': 'Unidade Local de Saúde de Trás-os-Montes e Alto Douro, EPE',
    'Centro Hospitalar Tâmega e Sousa, EPE': 'Unidade Local de Saúde do Tâmega e Sousa, EPE',
    'Centro Hospitalar Universitário Cova da Beira, EPE': 'Unidade Local de Saúde da Cova da Beira, EPE',
    'Centro Hospitalar Universitário Lisboa Central, EPE': 'Unidade Local de Saúde de São José, EPE',
    'Centro Hospitalar Universitário Lisboa Norte, EPE': 'Unidade Local de Saúde de Santa Maria, EPE',
    'Centro Hospitalar Universitário de Santo António, EPE': 'Unidade Local de Saúde de Santo António, EPE',
    'Centro Hospitalar Universitário do Porto, EPE': 'Unidade Local de Saúde de Santo António, EPE',
    'Centro Hospitalar Universitário de São João, EPE': 'Unidade Local de Saúde de São João, EPE',
    'Centro Hospitalar Universitário do Algarve, EPE': 'Unidade Local de Saúde do Algarve, EPE',
    'Centro Hospitalar Vila Nova de Gaia/Espinho, EPE': 'Unidade Local de Saúde de Vila Nova de Gaia/Espinho, EPE',
    'Centro Hospitalar de Leiria, EPE': 'Unidade Local de Saúde da Região de Leiria, EPE',
    'Centro Hospitalar de Lisboa Ocidental, EPE': 'Unidade Local de Saúde de Lisboa Ocidental, EPE',
    'Centro Hospitalar de Setúbal, EPE': 'Unidade Local de Saúde da Arrábida, EPE',
    'Centro Hospitalar do Baixo Vouga, EPE': 'Unidade Local de Saúde da Região de Aveiro, EPE',
    'Centro Hospitalar do Médio Ave, EPE': 'Unidade Local de Saúde do Médio Ave, EPE',
    'Centro Hospitalar do Oeste, EPE': 'Unidade Local de Saúde do Oeste, EPE',
    'Centro Hospitalar e Universitário de Coimbra, EPE': 'Unidade Local de Saúde de Coimbra, EPE',
    'Hospital Distrital da Figueira da Foz, EPE': 'Unidade Local de Saúde do Baixo Mondego, EPE',
    'Hospital Distrital de Santarém, EPE': 'Unidade Local de Saúde da Lezíria, EPE',
    'Hospital Espírito Santo de Évora, EPE': 'Unidade Local de Saúde do Alentejo Central, EPE',
    'Hospital Garcia de Orta, EPE': 'Unidade Local de Saúde de Almada-Seixal, EPE',
    'Hospital Professor Doutor Fernando Fonseca, EPE': 'Unidade Local de Saúde de Amadora/Sintra, EPE',
    'Hospital Santa Maria Maior, EPE': 'Unidade Local de Saúde de Barcelos/Esposende, EPE',
    'Hospital da Senhora da Oliveira, Guimarães, EPE': 'Unidade Local de Saúde do Alto Ave, EPE',
    'Hospital de Braga, EPE': 'Unidade Local de Saúde de Braga, EPE',
    'Hospital de Loures, EPE': 'Unidade Local de Saúde de Loures-Odivelas, EPE',
    'Hospital de Vila Franca de Xira, EPE': 'Unidade Local de Saúde do Estuário do Tejo, EPE',
    'Unidade Local de Saúde de Castelo Branco, EPE': 'Unidade Local de Saúde Castelo Branco, EPE',
    'Unidade Local de Saúde do Norte Alentejano, EPE': 'Unidade Local de Saúde do Alto Alentejo, EPE',
    'Instituto Português Oncologia de Coimbra, EPE': 'Instituto Português Oncologia F. Gentil - Centro, E.P.E.',
    'Instituto Português Oncologia de Lisboa, EPE': 'Instituto Português Oncologia F. Gentil - Lisboa, E.P.E.',
    'Instituto Português Oncologia do Porto, EPE': 'Instituto Português Oncologia F. Gentil - Porto, E.P.E.',
}

ocup['instituicao'] = ocup['Instituição'].replace(correspondencia)
urg['instituicao'] = urg['Instituição'].replace(correspondencia)

In [6]:
print(urg['instituicao'].nunique(), urg.groupby('instituicao').size().value_counts().to_dict())
print(ocup['instituicao'].nunique())
print(ocup.duplicated(['instituicao', 'Período']).sum(), urg.duplicated(['instituicao', 'Período']).sum())

40 {55: 40}
51
0 0


In [7]:
ocup[ocup['Instituição'].str.contains('Oncologia')].groupby('instituicao').size()

,0
instituicao,
"Instituto Português Oncologia F. Gentil - Centro, E.P.E.",31
"Instituto Português Oncologia F. Gentil - Lisboa, E.P.E.",31
"Instituto Português Oncologia F. Gentil - Porto, E.P.E.",31
"Instituto Português Oncologia F. Gentil - Centro, E.P.E.",24
"Instituto Português Oncologia F. Gentil - Lisboa, E.P.E.",24
"Instituto Português Oncologia F. Gentil - Porto, E.P.E.",24


In [8]:
def nome_ipo(nome):
    if 'Oncologia' not in nome:
        return nome
    if 'Coimbra' in nome or 'Centro' in nome:
        return 'Instituto Português Oncologia F. Gentil - Centro, E.P.E.'
    if 'Lisboa' in nome:
        return 'Instituto Português Oncologia F. Gentil - Lisboa, E.P.E.'
    if 'Porto' in nome:
        return 'Instituto Português Oncologia F. Gentil - Porto, E.P.E.'
    return nome

ocup['instituicao'] = ocup['instituicao'].apply(nome_ipo)
print(ocup['instituicao'].nunique())

48


**Conclusões da seção 3**

*O que foi feito:* em janeiro de 2024 muitas instituições mudaram de nome (centros hospitalares e hospitais passaram a Unidades Locais de Saúde). Foi criada a coluna `instituicao`, em que cada nome antigo é ligado ao nome atual, para que a série de cada instituição não fique cortada em duas. A coluna original `Instituição` foi mantida sem alterações.

*O que encontramos:*
- Na urgência, 74 nomes correspondem a 40 instituições: 7 mantêm o mesmo nome nos 55 meses e 33 mudam de nome em 2024-01.
- Depois da correspondência, as 40 instituições têm 55 meses cada e não há linhas duplicadas (uma linha por instituição e mês).
- Uma instituição tem três nomes ao longo do período (Centro Hospitalar Universitário do Porto, depois de Santo António, e por fim ULS de Santo António).
- Na ocupação, 85 nomes correspondem a 48 instituições: as 40 da urgência e 8 que só existem na ocupação (3 institutos de oncologia e 5 hospitais especializados ou de pequena dimensão).
- Os 3 institutos de oncologia exigiram um ajuste à parte, porque os nomes não coincidiam exatamente com os previstos.

*Limitações:* a correspondência foi construída com conhecimento externo sobre a reorganização de 2024, e por isso é validada na seção 4 comparando o número de camas antes e depois da mudança. Uma ULS nova pode ter um âmbito maior que o hospital antigo, o que deve ser considerado ao interpretar saltos em janeiro de 2024.

## 4. Teste das ligações
Comparação das camas em dezembro de 2023 e janeiro de 2024. Valores muito diferentes indicam uma ligação a conferir.

In [9]:
dez = ocup[ocup['Período'] == '2023-12'].set_index('instituicao')['Lotação Praticada']
jan = ocup[ocup['Período'] == '2024-01'].set_index('instituicao')['Lotação Praticada']

comparacao = pd.DataFrame({'dez_2023': dez, 'jan_2024': jan}).dropna()
comparacao['razao'] = (comparacao['jan_2024'] / comparacao['dez_2023']).round(2)
comparacao[(comparacao['razao'] < 0.7) | (comparacao['razao'] > 1.3)]

print(len(comparacao))

43


**Conclusões da seção 4**

- Foram comparadas as camas de 43 instituições em dezembro de 2023 (nome antigo) e janeiro de 2024 (nome novo). Nenhuma variou mais de 30%, o que indica que as ligações entre nomes são coerentes.
- O teste detecta ligações grosseiramente erradas, mas não a troca entre instituições de dimensão semelhante. A correspondência foi validada pelos dados e não por fonte oficial.

## 5. Valores errados
Valores identificados como erros de registo na compreensão dos dados são marcados como ausentes, para não distorcer os cálculos seguintes.

In [10]:
col = 'Nº Dias de Internamento'
loures = 'Unidade Local de Saúde de Loures-Odivelas, EPE'

# Loures, 2022: o valor 6360 aparece repetido em 8 meses
ocup.loc[(ocup['instituicao'] == loures) & (ocup['ano'] == 2022) & (ocup[col] == 6360), col] = None

# Loures-Odivelas, jan/2025: igual ao de fevereiro
ocup.loc[(ocup['instituicao'] == loures) & (ocup['Período'] == '2025-01') & (ocup[col] == 21057), col] = None

# Amadora/Sintra, jul/2026: igual ao de junho
ocup.loc[(ocup['instituicao'] == 'Unidade Local de Saúde de Amadora/Sintra, EPE') & (ocup['Período'] == '2026-07'), col] = None

print(ocup[col].isna().sum())

11


**Conclusões da seção 5**

*O que foi feito:* os valores identificados como erros de registo no notebook 01 foram marcados como ausentes na coluna de dias de internamento, sem os substituir por estimativas. A coluna original com a taxa de ocupação do portal não foi corrigida, porque será recalculada.

*Valores removidos (11 ausentes no total):*
- 1 já vinha ausente nos dados originais (Hospital de Cascais, outubro de 2022).
- 8 em Loures, em 2022: o valor 6.360 aparece repetido em oito meses, o que é incompatível com um valor acumulado.
- 1 em Loures-Odivelas, janeiro de 2025: igual ao de fevereiro, quando o ritmo dos números indica erro em janeiro.
- 1 em Amadora/Sintra, julho de 2026: igual ao de junho, o que sugere cópia do mês anterior.

*Limitações:*
- A identificação dos erros baseia-se no comportamento esperado de um valor acumulado, e não em confirmação da fonte.
- Os meses ausentes limitam a análise mensal de Loures em 2022.
- Os vazios das colunas de urgência (principalmente "sem triagem") não foram tratados, porque não se sabe se vazio significa zero ou dado não enviado.

## 6. Desacumulação
Os dias de internamento e os atendimentos são acumulados desde janeiro. O valor de cada mês é calculado como a diferença para o mês anterior (em janeiro, é o próprio valor). Se o mês anterior estiver em falta ou vazio, o resultado fica vazio.

In [11]:
# Ordenar por instituição e mês
ocup = ocup.sort_values(['instituicao', 'Período']).reset_index(drop=True)
urg = urg.sort_values(['instituicao', 'Período']).reset_index(drop=True)

def desacumular(df, coluna):
    g = df.groupby(['instituicao', 'ano'])
    dif = df[coluna] - g[coluna].shift(1)              # acumulado menos o anterior
    seguido = (df['mes'] - g['mes'].shift(1)) == 1     # o anterior é mesmo o mês anterior?
    mensal = dif.where(seguido)
    return mensal.where(df['mes'] != 1, df[coluna])    # janeiro = o próprio valor

# Ocupação
ocup['dias_mes'] = desacumular(ocup, 'Nº Dias de Internamento')

# Urgência (as 7 colunas de atendimentos)
colunas_atend = [c for c in urg.columns if c.startswith('Nº Atendimentos')]
nomes = ['vermelha', 'laranja', 'amarela', 'verde', 'azul', 'branca', 'sem_triagem']
for coluna, nome in zip(colunas_atend, nomes):
    urg[nome] = desacumular(urg, coluna)

In [12]:
ocup[(ocup['instituicao'] == 'Unidade Local de Saúde de Matosinhos, EPE') &
     (ocup['Período'] >= '2024-12') & (ocup['Período'] <= '2025-03')][
    ['Período', 'Nº Dias de Internamento', 'dias_mes']]

,Período,Nº Dias de Internamento,dias_mes
1289,2024-12-01,116899.0,10201.0
1290,2025-01-01,10481.0,10481.0
1291,2025-02-01,20354.0,9873.0
1292,2025-03-01,30672.0,10318.0


**Conclusões da seção 6**

*O que foi feito:* os dias de internamento e os atendimentos da urgência, que são valores acumulados desde janeiro, foram convertidos em valores de cada mês. O valor do mês é o acumulado menos o acumulado do mês anterior; em janeiro, que recomeça do zero, é o próprio valor. Os resultados foram guardados em colunas novas (`dias_mes`, `vermelha`, `laranja`, `amarela`, `verde`, `azul`, `branca` e `sem_triagem`), mantendo as colunas originais.

*Validação:* na ULS de Matosinhos, entre dezembro de 2024 e março de 2025, os valores mensais obtidos (10.201, 10.481, 9.873 e 10.318 dias) coincidem com os calculados manualmente na compreensão dos dados, e janeiro recomeça corretamente.

*Limitações:*
- Se o mês anterior estiver em falta ou ausente, o valor do mês fica ausente, em vez de ser estimado. Isso afeta sobretudo Loures em 2022, depois da remoção dos valores errados no passo anterior.
- Séries que começam a meio do ano não têm valor mensal no primeiro mês, por falta de referência.
- Os erros que o acumulado já trazia (quedas dentro do ano, principalmente em "sem triagem") passam a aparecer como valores mensais negativos, e serão verificados em seguida.

## 7. Verificação de valores negativos
Um valor mensal não pode ser negativo. Contagem dos casos que resultaram da desacumulação.

In [13]:
print("Ocupação:", (ocup['dias_mes'] < 0).sum())
print(urg[nomes].lt(0).sum())

Ocupação: 0
vermelha        1
laranja         0
amarela         0
verde           0
azul            3
branca          0
sem_triagem    18
dtype: int64


**Conclusões da seção 7**

*O que foi feito:* verificação de valores mensais negativos, que são impossíveis (não existe um número negativo de atendimentos ou de dias de internamento).

*O que encontramos:*
- Na ocupação não há valores negativos, porque os dois casos de queda identificados no notebook 01 (Hospital de Loures) já tinham sido removidos.
- Na urgência há 22 valores negativos: 1 na cor vermelha, 3 na azul e 18 em "sem triagem". Coincidem exatamente com as 22 quedas dentro do ano encontradas na compreensão dos dados, o que confirma que a desacumulação não introduziu erros novos.
- A coluna "sem triagem" concentra 18 dos 22 casos, tal como já concentrava a maioria dos valores ausentes, o que reforça que é a coluna menos fiável da tabela.

*Limitações:* um valor negativo indica que o acumulado desceu, mas não permite saber se o erro está no mês atual ou no anterior.

## 8. Tratamento dos valores negativos
Os valores mensais negativos da urgência são marcados como ausentes.

In [14]:
urg[nomes] = urg[nomes].where(urg[nomes] >= 0)
print(urg[nomes].lt(0).sum().sum())

0


**Conclusões da seção 8**

*O que foi feito:* os 22 valores mensais negativos da urgência foram marcados como ausentes, sem serem substituídos por estimativas. Depois do tratamento não resta nenhum valor negativo.

*Porquê:* um número negativo de atendimentos é impossível e indica um erro nos dados acumulados. Remover o valor é mais honesto do que o corrigir às cegas, porque não sabemos se o erro está no mês atual ou no anterior.

*Limitações:*
- São 22 valores em mais de 15 mil (menos de 0,2%), por isso o efeito nos resultados é pequeno.
- O mês anterior a cada negativo pode continuar ligeiramente inflado.
- A coluna "sem triagem" concentra a maior parte dos casos e deve ser interpretada com cautela.

## 9. Taxa de ocupação mensal
A taxa do portal é acumulada no ano e contém valores incoerentes. É recalculada para cada mês a partir dos dias de internamento do mês e das camas em funcionamento.

In [15]:
camas = ocup['Lotação Praticada'].where(ocup['Lotação Praticada'] > 0)
ocup['taxa_mes'] = (ocup['dias_mes'] / (camas * ocup['Período'].dt.days_in_month) * 100).round(1)

ocup['taxa_mes'].describe()

,taxa_mes
count,2442.000000
mean,89.377928
std,53.795629
min,15.800000
25%,79.800000
50%,86.700000
75%,93.600000
max,2611.300000


In [16]:
estranhas = ocup[(ocup['taxa_mes'] > 130) | (ocup['taxa_mes'] < 40)]
print(len(estranhas))
estranhas[['instituicao', 'Período', 'dias_mes', 'Lotação Praticada', 'taxa_mes']].sort_values('taxa_mes')

44


,instituicao,Período,dias_mes,Lotação Praticada,taxa_mes
194,Instituto Português Oncologia F. Gentil - Cent...,2024-10-01,636.0,130.0,15.8
700,"Unidade Local de Saúde da Região de Aveiro, EPE",2025-09-01,4322.0,440.0,32.7
1946,"Unidade Local de Saúde do Arco Ribeirinho, EPE",2024-09-01,4150.0,351.0,39.4
705,"Unidade Local de Saúde da Região de Aveiro, EPE",2026-02-01,4509.0,407.0,39.6
2442,"Unidade Local de Saúde do Tâmega e Sousa, EPE",2024-10-01,19464.0,481.0,130.5
2444,"Unidade Local de Saúde do Tâmega e Sousa, EPE",2024-12-01,19506.0,481.0,130.8
902,"Unidade Local de Saúde de Barcelos/Esposende, EPE",2024-03-01,4021.0,99.0,131.0
2451,"Unidade Local de Saúde do Tâmega e Sousa, EPE",2025-07-01,19648.0,481.0,131.8
2421,"Unidade Local de Saúde do Tâmega e Sousa, EPE",2023-01-01,19780.0,481.0,132.7
2423,"Unidade Local de Saúde do Tâmega e Sousa, EPE",2023-03-01,19805.0,481.0,132.8


In [17]:
ocup[ocup['instituicao'] == 'Unidade Local de Saúde do Algarve, EPE'][['Período', 'dias_mes', 'Lotação Praticada', 'taxa_mes']].tail(20)

,Período,dias_mes,Lotação Praticada,taxa_mes
1729,2024-12-01,24653.0,905.0,87.9
1730,2025-01-01,26759.0,918.0,94.0
1731,2025-02-01,24934.0,918.0,97.0
1732,2025-03-01,27291.0,911.0,96.6
1733,2025-04-01,26025.0,908.0,95.5
1734,2025-05-01,26617.0,905.0,94.9
1735,2025-06-01,25017.0,903.0,92.3
1736,2025-07-01,26024.0,903.0,93.0
1737,2025-08-01,25491.0,903.0,91.1
1738,2025-09-01,25163.0,902.0,93.0


In [18]:
ocup[(ocup['instituicao'] == 'Unidade Local de Saúde de Matosinhos, EPE') &
     (ocup['Período'] >= '2024-12') & (ocup['Período'] <= '2025-03')][
    ['Período', 'dias_mes', 'Lotação Praticada', 'taxa_mes']]

,Período,dias_mes,Lotação Praticada,taxa_mes
1289,2024-12-01,10201.0,382.0,86.1
1290,2025-01-01,10481.0,385.0,87.8
1291,2025-02-01,9873.0,385.0,91.6
1292,2025-03-01,10318.0,385.0,86.5


**Conclusões da seção 9**

*O que foi feito:* a taxa de ocupação do portal é acumulada no ano e tem valores incoerentes, por isso foi recalculada para cada mês: dias de internamento do mês ÷ (camas × dias do mês) × 100. Na ULS de Matosinhos, os valores obtidos (86,1%, 87,8%, 91,6% e 86,5%) coincidem com o cálculo manual.

*O que encontramos:*
- A mediana é 86,7% e metade dos valores está entre 79,8% e 93,6%, valores plausíveis.
- 44 linhas (1,8%) ficam fora do intervalo de 40% a 130%, com máximo de 2.611% (Centro de Reabilitação Rovisco Pais, 2022-01, com 2 camas registadas).
- 29 dessas linhas vêm de duas instituições. A ULS do Tâmega e Sousa tem taxa persistentemente alta (130% a 175% em 22 meses), o que sugere número de camas registado inferior às camas utilizadas. Na ULS do Algarve, a lotação passa de cerca de 920 camas em dezembro de 2025 para 329 em janeiro de 2026, enquanto os dias de internamento se mantêm estáveis (25 a 26 mil por mês), o que faz a taxa subir para cerca de 255% a 266%. A causa da mudança não está documentada.
- Os restantes são casos isolados, alguns com um mês muito baixo seguido de um muito alto, padrão compatível com erro num valor acumulado.

*Limitações:* a taxa depende da "Lotação Praticada", cujo significado não está documentado pelo portal. Taxas acima de 100% podem ocorrer, mas as persistentemente elevadas devem ser interpretadas com cautela.

## 10. Marcação das taxas pouco confiáveis
As linhas com taxa mensal fora do intervalo de 40% a 130% são marcadas como pouco confiáveis, sem serem removidas.

In [19]:
ocup['taxa_fiavel'] = ocup['taxa_mes'].between(40, 130)
print(ocup['taxa_fiavel'].value_counts())

taxa_fiavel
True     2398
False      66
Name: count, dtype: int64


**Conclusões da seção 10**

- 66 linhas ficam marcadas como pouco confiáveis (44 com taxa fora de 40%-130% e 22 sem taxa). Nenhuma foi removida, porque os dias de internamento dessas linhas podem estar corretos.
- Na análise da taxa de ocupação, só serão usadas as linhas confiáveis. Para o Algarve em 2026, será usada a evolução dos dias de internamento, e não a taxa.

## 11. Junção das tabelas
As tabelas de ocupação e urgência são unidas pela instituição e pelo mês. São mantidas todas as linhas das duas tabelas, mesmo quando só uma delas tem dados.

In [20]:
tabela = ocup[['instituicao', 'Região', 'Período', 'dias_mes', 'Lotação Praticada', 'taxa_mes', 'taxa_fiavel']].merge(
    urg[['instituicao', 'Período', 'Região'] + nomes],
    on=['instituicao', 'Período'], how='outer', suffixes=('', '_urg'), validate='one_to_one')

tabela['regiao'] = tabela['Região'].fillna(tabela['Região_urg'])
tabela['ano'] = tabela['Período'].dt.year
tabela['mes'] = tabela['Período'].dt.month
tabela['taxa_fiavel'] = tabela['taxa_fiavel'].eq(True)

tabela = tabela.drop(columns=['Região', 'Região_urg']).rename(
    columns={'Período': 'periodo', 'Lotação Praticada': 'camas'})
tabela = tabela.sort_values(['instituicao', 'periodo']).reset_index(drop=True)

print(tabela.shape)

(2471, 16)


**Conclusões da seção 11**

*O que foi feito:* as tabelas de ocupação e urgência foram unidas pela instituição (nome atual) e pelo mês, numa única tabela com uma linha por instituição e mês. Foram mantidas todas as linhas das duas tabelas.

*O que encontramos:*
- A tabela final tem 2.471 linhas e 16 colunas: as 2.464 linhas da ocupação mais 7 meses do Hospital de Loures que existem na urgência e faltam na ocupação.
- Não há linhas duplicadas (a junção foi feita com verificação de correspondência um-para-um).
- As 8 instituições que só existem na ocupação ficam com as colunas de urgência vazias, e os 7 meses de Loures ficam com a ocupação vazia.
- As colunas finais são: instituicao, regiao, periodo, ano, mes, dias_mes, camas, taxa_mes, taxa_fiavel e as sete colunas mensais da urgência (vermelha, laranja, amarela, verde, azul, branca, sem_triagem).

*Limitações:* os vazios que vêm de meses em falta ou de valores removidos foram mantidos como vazios, sem estimativa.

## 12. Gravação da tabela limpa
A tabela final é guardada em CSV, para ser usada no notebook de análise.

In [21]:
caminho = '/content/drive/MyDrive/semantix/dados_limpos.csv'
tabela.to_csv(caminho, index=False, encoding='utf-8-sig')

# Conferência: ler o arquivo que acabou de ser salvo
print(pd.read_csv(caminho).shape)

(2471, 16)


**Conclusões da seção 12**

- A tabela final foi gravada em `dados_limpos.csv` (2.471 linhas e 16 colunas) e relida com sucesso, o que confirma que o arquivo está íntegro.
- Os arquivos originais não foram alterados: toda a limpeza foi feita no código e pode ser reproduzida executando o notebook de cima para baixo.

## Conclusões do notebook 02

*Resultado:* tabela única com uma linha por instituição e mês (48 instituições, 2022-01 a 2026-07), com valores mensais de ocupação e de urgência.

*O que foi feito:*
- Período convertido para data e nomes padronizados.
- Nomes antigos ligados aos atuais (mudanças de janeiro de 2024), validados pelo número de camas.
- Valores acumulados convertidos em valores mensais.
- 10 valores de dias de internamento (erros de registo) e 22 valores negativos da urgência marcados como ausentes, sem estimativas. Somados a 1 valor que já vinha vazio, a coluna de dias de internamento tem 11 ausentes.
- Taxa de ocupação mensal recalculada (mediana de 86,7%). Das 2.464 linhas de ocupação, 66 ficam marcadas como pouco confiáveis (44 fora de 40%-130% e 22 sem taxa).

*Limitações:*
- Na ULS do Algarve, a lotação cai de cerca de 920 para 329 camas em janeiro de 2026 sem que os dias de internamento mudem, por isso a taxa de 2026 não é confiável; nessa região, usar os dias de internamento.
- Os vazios da urgência ("sem triagem" e "Branca") não foram tratados, porque não se sabe se vazio significa zero ou dado não enviado.
- A correspondência de nomes foi validada pelos dados e não por fonte oficial.
- As coordenadas do portal estão arredondadas em parte das linhas e não foram usadas.

*Próximo passo:* notebook 03, análise das três perguntas do projeto.

### Pontos em aberto

Questões identificadas na compreensão dos dados e o seu encaminhamento.

| # | Pendência | Onde foi tratada | Status |
|---|---|---|---|
| 1 | Dias de internamento acumulados? | Notebook 01, seção 4.7 | Resolvido: sim, acumulados desde janeiro |
| 2 | Atendimentos da urgência acumulados? | Notebook 01, seção 4.7 | Resolvido: sim, acumulados desde janeiro |
| 3 | Valores extremos da taxa de ocupação (máximo de 3.751,6%) | Notebook 02, seções 9 e 10 | Resolvido: taxa mensal recalculada e linhas pouco confiáveis marcadas |
| 4 | Mudanças de nome em 2024-01 | Notebook 02, seções 3 e 4 | Resolvido: correspondência criada e validada pelo número de camas |
| 5 | 2 nomes diferem só na escrita; 11 instituições só existem na ocupação | Notebook 02, seções 2 e 11 | Resolvido: grafias padronizadas; as outras ficam sem dados de urgência |
| 6 | Significado da cor Branca na triagem | Notebook 03, pergunta 2 | Parcialmente resolvido: fontes jornalísticas indicam que não é nível de gravidade; falta fonte oficial |
| 7 | Período coberto (2022-01 a 2026-07, 55 meses nas duas tabelas) | Notebook 01, seção 4.5 | Resolvido |
| 8 | Vazios em "sem triagem" e "Branca" (9 instituições) | Sem tratamento | Aberto: não se sabe se vazio significa zero ou dado não enviado |
| 9 | Converter tipos | Notebook 02, seção 1 | Resolvido: Período convertido em data; contagens ficam como decimais por causa dos vazios |
| 10 | Hospital de Loures com 4 meses de 2023 sem lotação e sem taxa | Notebook 02, seções 9 e 10 | Resolvido: ficam sem taxa e marcados como pouco confiáveis; causa desconhecida |
| 11 | Meses em falta na ocupação (Loures 17/24, Arcebispo João Crisóstomo 21, Magalhães Lemos 13) | Notebook 02, seção 11 | Aberto: mantidos como vazios; registar como limitação |
| 12 | Possível mudança de âmbito das instituições em 2024-01 | Análise e README | Aberto |
| 13 | Validar o acumulado em todas as instituições | Notebook 01, seção 4.7 | Resolvido: 2 quedas na ocupação e 22 na urgência |
| 14 | Loures 2022: valor 6.360 repetido em 8 meses | Notebook 02, seção 5 | Resolvido: marcado como ausente |
| 15 | Valores repetidos na ocupação (2 casos adicionais) | Notebook 02, seção 5 | Resolvido: marcados como ausentes |
| 16 | Valores mensais negativos na urgência (22 casos) | Notebook 02, seção 8 | Resolvido: marcados como ausentes |
| 17 | Taxa mensal incoerente em 44 linhas | Notebook 02, seção 10 | Resolvido: marcadas na coluna taxa_fiavel |
| 18 | Coordenadas arredondadas em cerca de 11% das linhas | Sem tratamento | Aberto: não usadas; corrigir só se houver mapa |
| 19 | Algarve: lotação cai de cerca de 920 para 329 camas em 2026 | Notebook 02, seção 9 | Aberto: usar dias de internamento em vez da taxa |
| 20 | Significado de "Lotação Praticada" | Sem tratamento | Aberto: depende de fonte oficial |